In [0]:
# Turn on Change Data Feed (CDF) for row level change tracking

spark.sql("ALTER TABLE rag_pipeline.main.silver_embeddings_fixed_size SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql("ALTER TABLE rag_pipeline.main.silver_embeddings_section_aware SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

DataFrame[]

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-12537af8-799e-44b5-a0c1-28e31bb36930
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-12537af8-799e-44b5-a0c1-28e31bb36930
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-12537af8-799e-44b5-

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

def embed_query(text):
    response = client.predict(
        endpoint="databricks-bge-large-en",
        inputs={"input": [text]}
    )
    return response["data"][0]["embedding"]

query_text = "What are NVIDIA's supply chain risk factors?"
query_vector = embed_query(query_text)

# Query the fixed-size index
fixed_index = vsc.get_index(endpoint_name="rag_pipeline_endpoint", index_name="rag_pipeline.main.fixed_size_index")
fixed_results = fixed_index.similarity_search(
    query_vector=query_vector,
    columns=["chunk_id", "company", "fiscal_year", "chunk_text"],
    num_results=5
)
display(fixed_results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 5,
  'columns': [{'name': 'chunk_id'},
   {'name': 'company'},
   {'name': 'fiscal_year'},
   {'name': 'chunk_text'},
   {'name': 'score'}]},
 'result': {'row_count': 5,
  'data_array': [[5711.0,
    'NVDA',
    '2024',
    "es/edgar/data/1045810/000104581025000023/nvda-20250126.htm 26/118\nTable of Contents\n• purchasing decisions made, and inventory levels held by, distributors, ODMs, OEMs, system integrators, other channel partners\nand other third parties;\n• the ability of developers, end customers and other third parties to build, enhance, and maintain accelerated computing\napplications that leverage our platforms;\n• the availability of third-party content on our platforms, such as GeForce NOW;\n• the demand for accelerated computing, AI-related cloud services, or large language models;\n• changes that impact the ecosystem for the architectures underlying our products and technologies;\n• government actions or changes in governmental policies, such

In [0]:
# Section Aware Vector Search

section_index = vsc.get_index(endpoint_name="rag_pipeline_endpoint", index_name="rag_pipeline.main.section_aware_index")
section_results = section_index.similarity_search(
    query_vector=query_vector,
    columns=["chunk_id", "company", "fiscal_year", "chunk_text"],
    num_results=5
)
display(section_results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 5,
  'columns': [{'name': 'chunk_id'},
   {'name': 'company'},
   {'name': 'fiscal_year'},
   {'name': 'chunk_text'},
   {'name': 'score'}]},
 'result': {'row_count': 5,
  'data_array': [[6571.0,
    'NVDA',
    '2024',
    ' GPUs and semiconductors associated with AI have subjected and\nmay in the future subject downstream users of our products to additional restrictions on the use, resale, repair, or transfer of our\nproducts, negatively impacting our business and ﬁnancial results. Controls could negatively impact our cost and/or ability to provide\nservices such as NVIDIA AI cloud services and could impact the cost and/or ability for our CSPs and customers to provide services to\ntheir end customers, even outside China.\nExport controls could disrupt our supply chain and distribution channels, negatively impacting our ability to serve demand, including in\nmarkets outside China and for our gaming products. The possibility of additional export controls h